In [ ]:
"""
Backfill Naver Detail - [이미지/미디어 포함]으로 저장된 Naver 이슈들의 detail을 RawLog 기반으로 복원

사용법:
    python backfill-naver-detail.py [일수]
    예: python backfill-naver-detail.py 3  # 최근 3일간의 이슈 복원
"""

import json
import sqlite3
import sys
import os
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Tuple, List

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# 배치 처리 크기
BATCH_SIZE = 100


def find_database_path() -> str:
    """
    데이터베이스 파일 경로를 안전하게 찾기
    
    Returns:
        str: 데이터베이스 파일의 절대 경로
        
    Raises:
        FileNotFoundError: 데이터베이스 파일을 찾을 수 없을 때
    """
    possible_paths = [
        'prisma/dev.db',
        os.path.join('..', 'prisma', 'dev.db'),
        os.path.join(os.path.dirname(__file__), '..', 'prisma', 'dev.db'),
        os.path.join(Path(__file__).parent.parent, 'prisma', 'dev.db'),
    ]
    
    for path in possible_paths:
        abs_path = os.path.abspath(path)
        if os.path.exists(abs_path):
            logger.info(f"데이터베이스 경로: {abs_path}")
            return abs_path
    
    raise FileNotFoundError(
        f"데이터베이스 파일을 찾을 수 없습니다. 다음 경로를 확인했습니다: {possible_paths}"
    )


def find_matching_rawlog(
    cur: sqlite3.Cursor,
    external_post_id: Optional[str],
    source_url: Optional[str]
) -> Optional[sqlite3.Row]:
    """
    RawLog에서 매칭되는 항목 찾기 (json_extract 사용하여 성능 최적화)
    
    Args:
        cur: 데이터베이스 커서
        external_post_id: 외부 게시글 ID
        source_url: 소스 URL
        
    Returns:
        Optional[sqlite3.Row]: 매칭된 RawLog 행 또는 None
    """
    # externalPostId로 검색
    if external_post_id:
        try:
            cur.execute("""
                SELECT id, content, metadata, createdAt
                FROM RawLog
                WHERE source = 'naver'
                  AND json_extract(metadata, '$.externalPostId') = ?
                ORDER BY createdAt DESC
                LIMIT 5
            """, (str(external_post_id),))
            
            candidates = cur.fetchall()
            for candidate in candidates:
                try:
                    meta = json.loads(candidate["metadata"]) if candidate["metadata"] else {}
                    meta_post_id = meta.get("externalPostId")
                    if meta_post_id and str(meta_post_id) == str(external_post_id):
                        return candidate
                except (json.JSONDecodeError, TypeError) as e:
                    logger.debug(f"메타데이터 파싱 실패 (RawLog ID: {candidate['id']}): {e}")
                    continue
        except sqlite3.OperationalError as e:
            # SQLite 버전이 낮거나 JSON 함수 미지원 시 fallback
            logger.warning(f"json_extract 사용 실패, LIKE 검색으로 fallback: {e}")
            # Fallback to LIKE search for older SQLite versions
            cur.execute("""
                SELECT id, content, metadata, createdAt
                FROM RawLog
                WHERE source = 'naver'
                  AND metadata LIKE ?
                ORDER BY createdAt DESC
                LIMIT 20
            """, (f'%"externalPostId":"{external_post_id}"%',))
            
            candidates = cur.fetchall()
            for candidate in candidates:
                try:
                    meta = json.loads(candidate["metadata"]) if candidate["metadata"] else {}
                    meta_post_id = meta.get("externalPostId")
                    if meta_post_id and str(meta_post_id) == str(external_post_id):
                        return candidate
                except (json.JSONDecodeError, TypeError):
                    continue
    
    # sourceUrl로 검색
    if source_url:
        try:
            cur.execute("""
                SELECT id, content, metadata, createdAt
                FROM RawLog
                WHERE source = 'naver'
                  AND json_extract(metadata, '$.url') = ?
                ORDER BY createdAt DESC
                LIMIT 5
            """, (str(source_url),))
            
            candidates = cur.fetchall()
            for candidate in candidates:
                try:
                    meta = json.loads(candidate["metadata"]) if candidate["metadata"] else {}
                    meta_url = meta.get("url")
                    if meta_url and str(meta_url) == str(source_url):
                        return candidate
                except (json.JSONDecodeError, TypeError) as e:
                    logger.debug(f"메타데이터 파싱 실패 (RawLog ID: {candidate['id']}): {e}")
                    continue
        except sqlite3.OperationalError as e:
            logger.warning(f"json_extract 사용 실패, LIKE 검색으로 fallback: {e}")
            # Fallback to LIKE search
            cur.execute("""
                SELECT id, content, metadata, createdAt
                FROM RawLog
                WHERE source = 'naver'
                  AND metadata LIKE ?
                ORDER BY createdAt DESC
                LIMIT 20
            """, (f'%"url":"{source_url}"%',))
            
            candidates = cur.fetchall()
            for candidate in candidates:
                try:
                    meta = json.loads(candidate["metadata"]) if candidate["metadata"] else {}
                    meta_url = meta.get("url")
                    if meta_url and str(meta_url) == str(source_url):
                        return candidate
                except (json.JSONDecodeError, TypeError):
                    continue
    
    return None


def process_issues(
    cur: sqlite3.Cursor,
    issues: List[sqlite3.Row]
) -> Tuple[int, int, int, List[Tuple[str, str, str]]]:
    """
    이슈 목록을 처리하고 업데이트할 데이터를 수집
    
    Args:
        cur: 데이터베이스 커서
        issues: 처리할 이슈 목록
        
    Returns:
        Tuple[업데이트된 수, RawLog 매칭 실패 수, 본문 없음 수, 업데이트 데이터 목록]
    """
    updated = 0
    skipped_no_raw = 0
    skipped_no_content = 0
    update_batch = []  # [(raw_content, updated_at, issue_id), ...]
    
    for idx, row in enumerate(issues, 1):
        # sqlite3.Row를 딕셔너리로 변환하여 get() 메서드 사용 가능하게 함
        row_dict = dict(row)
        issue_id = row_dict["id"]
        external_post_id = row_dict.get("externalPostId")
        source_url = row_dict.get("sourceUrl")
        summary = row_dict.get("summary", "")[:40] if row_dict.get("summary") else ""
        
        logger.info(f"[{idx}/{len(issues)}] 이슈 처리: {issue_id} | {summary}")
        
        if not external_post_id and not source_url:
            logger.warning(f"  ⚠️ externalPostId와 sourceUrl이 모두 없습니다. 스킵합니다.")
            skipped_no_raw += 1
            continue
        
        # RawLog 매칭
        matched_raw = find_matching_rawlog(cur, external_post_id, source_url)
        
        if not matched_raw:
            logger.warning(f"  ❌ 매칭되는 RawLog를 찾지 못했습니다.")
            skipped_no_raw += 1
            continue
        
        raw_content = (matched_raw["content"] or "").strip()
        if not raw_content or raw_content == '[이미지/미디어 포함]':
            logger.warning(f"  ⚠️ RawLog에도 유효한 본문이 없습니다.")
            skipped_no_content += 1
            continue
        
        # 업데이트 배치에 추가
        update_batch.append((
            raw_content,
            datetime.utcnow().isoformat(),
            issue_id
        ))
        updated += 1
        logger.info(f"  ✅ detail 업데이트 예정 (RawLog ID: {matched_raw['id']})")
    
    return updated, skipped_no_raw, skipped_no_content, update_batch


def batch_update(
    conn: sqlite3.Connection,
    cur: sqlite3.Cursor,
    update_batch: List[Tuple[str, str, str]],
    batch_size: int = BATCH_SIZE
) -> int:
    """
    배치 단위로 업데이트 실행
    
    Args:
        conn: 데이터베이스 연결
        cur: 데이터베이스 커서
        update_batch: 업데이트할 데이터 목록
        batch_size: 배치 크기
        
    Returns:
        int: 에러 발생 건수
    """
    error_count = 0
    
    for i in range(0, len(update_batch), batch_size):
        batch = update_batch[i:i + batch_size]
        try:
            cur.executemany(
                "UPDATE ReportItemIssue SET detail = ?, updatedAt = ? WHERE id = ?",
                batch
            )
            conn.commit()
            logger.info(f"배치 커밋 완료: {len(batch)}개 업데이트 (전체 {i + len(batch)}/{len(update_batch)})")
        except Exception as e:
            conn.rollback()
            logger.error(f"배치 커밋 실패: {e}", exc_info=True)
            error_count += len(batch)
    
    return error_count


def main(days: int = 3) -> None:
    """
    메인 함수
    
    Args:
        days: 처리할 일수 (기본값: 3일)
    """
    logger.info(f"🛠 최근 {days}일 동안의 이슈 중 [이미지/미디어 포함] 본문을 복원합니다.")
    
    # 데이터베이스 연결
    db_path = find_database_path()
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    
    try:
        cur = conn.cursor()
        
        # 날짜 기반 필터링 (JavaScript 버전과 일관성 유지)
        since_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        logger.info(f"조회 기간: {since_date} ~ 오늘")
        
        # 대상 이슈 조회
        cur.execute("""
            SELECT id, summary, detail, externalPostId, sourceUrl, date
            FROM ReportItemIssue
            WHERE detail = '[이미지/미디어 포함]'
              AND date >= ?
            ORDER BY createdAt DESC
        """, (since_date,))
        
        issues = cur.fetchall()
        logger.info(f"대상 이슈 수: {len(issues)}")
        
        if len(issues) == 0:
            logger.info("✅ 업데이트할 이슈가 없습니다.")
            return
        
        # 이슈 처리 및 업데이트 데이터 수집
        updated, skipped_no_raw, skipped_no_content, update_batch = process_issues(cur, issues)
        
        # 배치 업데이트 실행
        error_count = 0
        if update_batch:
            error_count = batch_update(conn, cur, update_batch)
        
        # 결과 요약
        logger.info("\n" + "=" * 50)
        logger.info("결과 요약")
        logger.info("=" * 50)
        logger.info(f"총 대상 이슈: {len(issues)}")
        logger.info(f"  ✅ 업데이트된 이슈: {updated}")
        logger.info(f"  ❌ RawLog 매칭 실패: {skipped_no_raw}")
        logger.info(f"  ⚠️ RawLog에 유효한 본문 없음: {skipped_no_content}")
        if error_count > 0:
            logger.error(f"  ❌ 업데이트 에러: {error_count}")
        logger.info("=" * 50)
        
    except Exception as e:
        conn.rollback()
        logger.error(f"❌ 오류 발생: {e}", exc_info=True)
        raise
    finally:
        conn.close()
        logger.info("완료되었습니다.")


if __name__ == "__main__":
    # 명령행 인자 처리
    days = 3  # 기본값
    if len(sys.argv) > 1:
        try:
            days = int(sys.argv[1])
            if days <= 0:
                logger.error("❌ 일수는 1 이상의 숫자여야 합니다.")
                sys.exit(1)
        except ValueError:
            logger.error("❌ 일수는 숫자여야 합니다.")
            sys.exit(1)
    
    main(days)